In [3]:
# ==========================================================
# Cell 1 : Import Required Libraries
# ==========================================================

import os
import warnings
import joblib

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

warnings.filterwarnings("ignore")

# Create models folder
os.makedirs("../models", exist_ok=True)

# Load Dataset
df = pd.read_csv("../data/processed/retail_features.csv")

print("Dataset Loaded Successfully")
print("Shape :", df.shape)

display(df.head())

Dataset Loaded Successfully
Shape : (20000, 57)


,Order_ID,Order_Date,Customer_ID,Product_ID,Quantity,Unit_Price,Discount,Sales,Cost,Profit,...,Sub_Category_Snacks,Sub_Category_Sofas,Sub_Category_Tables,Sub_Category_Women,Weekday_Monday,Weekday_Saturday,Weekday_Sunday,Weekday_Thursday,Weekday_Tuesday,Weekday_Wednesday
0,ORD100000,2024-05-10,CUST01487,PROD0029,7,29.35,0.26,152.03,98.82,53.21,...,0,0,0,0,0,0,0,0,0,0
1,ORD100001,2024-11-10,CUST00042,PROD0026,7,247.81,0.07,1613.24,1048.61,564.63,...,1,0,0,0,0,0,1,0,0,0
2,ORD100002,2022-05-02,CUST01197,PROD0070,1,365.99,0.09,333.05,216.48,116.57,...,0,0,0,0,1,0,0,0,0,0
3,ORD100003,2023-04-12,CUST00679,PROD0011,2,269.54,0.16,452.83,294.34,158.49,...,0,0,0,0,0,0,0,0,0,1
4,ORD100004,2022-11-27,CUST01274,PROD0100,5,163.63,0.13,711.79,462.66,249.13,...,0,0,0,0,0,0,1,0,0,0


In [4]:
# ==========================================================
# Cell 2 : Feature Selection
# ==========================================================

target = "Profit"

drop_columns = [
    "Order_ID",
    "Order_Date",
    "Customer_ID",
    "Product_ID",
    "Profit",
    "Sales"
]

drop_columns = [col for col in drop_columns if col in df.columns]

X = df.drop(columns=drop_columns)

# Keep only numeric features
X = X.select_dtypes(include=[np.number])

# Fill missing values
X = X.fillna(0)

y = df[target]

print("Feature Matrix :", X.shape)
print("Target Vector  :", y.shape)

Feature Matrix : (20000, 51)
Target Vector  : (20000,)


In [5]:
# ==========================================================
# Cell 3 : Train Test Split
# ==========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training Samples :", X_train.shape)
print("Testing Samples  :", X_test.shape)

Training Samples : (16000, 51)
Testing Samples  : (4000, 51)


In [6]:
# ==========================================================
# Cell 4 : Linear Regression
# ==========================================================

lr = LinearRegression()

lr.fit(X_train, y_train)

print("Linear Regression Model Trained")

Linear Regression Model Trained


In [7]:
# ==========================================================
# Cell 5 : Random Forest
# ==========================================================

rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

print("Random Forest Model Trained")

Random Forest Model Trained


In [8]:
# ==========================================================
# Cell 6 : XGBoost
# ==========================================================

xgb = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    objective="reg:squarederror"
)

xgb.fit(X_train, y_train)

print("XGBoost Model Trained")

XGBoost Model Trained


In [9]:
# ==========================================================
# Cell 7 : LightGBM
# ==========================================================

lgbm = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    random_state=42
)

lgbm.fit(X_train, y_train)

print("LightGBM Model Trained")

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003686 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1714
[LightGBM] [Info] Number of data points in the train set: 16000, number of used features: 50
[LightGBM] [Info] Start training from score 379.193341


  File "C:\Users\Shree\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\Shree\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Shree\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\Shree\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


LightGBM Model Trained


In [10]:
# ==========================================================
# Cell 8 : Save Models
# ==========================================================

joblib.dump(lr, "../models/linear_regression.pkl")
joblib.dump(rf, "../models/random_forest.pkl")
joblib.dump(xgb, "../models/xgboost.pkl")
joblib.dump(lgbm, "../models/lightgbm.pkl")

print("All Models Saved Successfully!")

All Models Saved Successfully!


In [11]:
# ==========================================================
# Cell 9 : Save Test Dataset
# ==========================================================

X_test.to_csv("../data/processed/X_test.csv", index=False)

y_test.to_frame(name="Profit").to_csv(
    "../data/processed/y_test.csv",
    index=False
)

print("Test Dataset Saved Successfully!")

Test Dataset Saved Successfully!


In [12]:
# ==========================================================
# Cell 10 : Summary
# ==========================================================

print("=" * 60)
print("MODEL BUILDING COMPLETED SUCCESSFULLY")
print("=" * 60)

print("\nModels Created")
print("------------------------")
print("✔ Linear Regression")
print("✔ Random Forest")
print("✔ XGBoost")
print("✔ LightGBM")

print("\nFiles Saved")
print("------------------------")
print("../models/linear_regression.pkl")
print("../models/random_forest.pkl")
print("../models/xgboost.pkl")
print("../models/lightgbm.pkl")
print("../data/processed/X_test.csv")
print("../data/processed/y_test.csv")

MODEL BUILDING COMPLETED SUCCESSFULLY

Models Created
------------------------
✔ Linear Regression
✔ Random Forest
✔ XGBoost
✔ LightGBM

Files Saved
------------------------
../models/linear_regression.pkl
../models/random_forest.pkl
../models/xgboost.pkl
../models/lightgbm.pkl
../data/processed/X_test.csv
../data/processed/y_test.csv


In [13]:
!pip install scikit-learn xgboost lightgbm joblib